# FLUX.1 Image Generator on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hirannalaka19/omnivoice-colab/blob/main/FLUX_Colab.ipynb)

Generate images with **[FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)** from
Black Forest Labs, in a Gradio UI, on a Colab **A100** GPU.

---

## Before you start (one time, 2 minutes)

1. **Pick the GPU** - `Runtime` -> `Change runtime type` -> **A100 GPU** -> *Save*.
   A100 needs Colab Pro. L4 and T4 also work; the app drops to CPU offload or 4-bit automatically.
2. **Accept the FLUX licence** - open <https://huggingface.co/black-forest-labs/FLUX.1-dev>
   while signed in to Hugging Face and click *Agree and access repository*. The model is gated,
   so this is required.
3. **Add your Hugging Face token to Colab Secrets** - create a **Read** token at
   <https://huggingface.co/settings/tokens>, then click the **key icon** in the left sidebar ->
   *Add new secret* -> Name: `HF_TOKEN`, Value: your token -> switch on **Notebook access**.
   It persists across all future sessions.

## Then just run the cells in order

`1` check GPU -> `2` install -> `3` download weights -> `4` run the app, and click the
`https://....gradio.live` link that appears.

> Prefer no licence and no token? Choose `black-forest-labs/FLUX.1-schnell` in step 3 -
> it is Apache-2.0, ungated, and needs only 4 steps per image.

In [ ]:
#@title 1. Check the GPU { display-mode: "form" }
#@markdown Runtime -> Change runtime type -> **A100 GPU**, then run this cell.

!nvidia-smi

import shutil

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU attached. Runtime -> Change runtime type -> GPU (A100 recommended), then re-run."
    )

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_disk = shutil.disk_usage("/content").free / 1024**3

print(f"\nGPU:        {name}  -  {vram:.0f} GB VRAM")
print(f"Free disk:  {free_disk:.0f} GB  (FLUX.1-dev weights need ~34 GB)")
print(f"torch:      {torch.__version__}")

if vram >= 34:
    print("\nMode: bf16, whole pipeline on the GPU  ->  ~10-15 s per 1024x1024 image.")
elif vram >= 20:
    print("\nMode: bf16 + CPU offload  ->  ~50-70 s per image. Switch to A100 for speed.")
else:
    print("\nMode: 4-bit NF4  ->  ~2-4 min per image. It works, but A100 or L4 is much nicer.")

if free_disk < 40:
    print("\nWARNING: less than 40 GB of free disk - the download may fail part way through.")

In [ ]:
#@title 2. Install FLUX + the app { display-mode: "form" }
#@markdown Pulls diffusers, gradio and friends. Colab's PyTorch is left alone on purpose. Takes ~2 minutes.

%cd /content/
!rm -rf ./omnivoice-colab
!git clone https://github.com/hirannalaka19/omnivoice-colab.git
%cd ./omnivoice-colab
!pip install -r flux_colab.txt

from IPython.display import clear_output

clear_output()

import importlib

missing = []
for module in ("diffusers", "transformers", "accelerate", "gradio", "peft", "huggingface_hub"):
    try:
        version = importlib.import_module(module).__version__
        print(f"{module:<18} {version}")
    except Exception as err:
        missing.append(f"{module} ({err})")

if missing:
    print("\nSomething did not install:", ", ".join(missing))
    print("Re-run this cell. If it keeps failing: Runtime -> Restart session, then run it again.")
else:
    print("\nInstall complete. Run cell 3 next.")

In [ ]:
#@title 3. Download the model weights { display-mode: "form" }
#@markdown One time per session. FLUX.1-dev is ~34 GB, so expect a few minutes.
#@markdown FLUX.1-dev and FLUX.1-Krea-dev are **gated** - accept the licence on the model page
#@markdown and add your `HF_TOKEN` to Colab Secrets (key icon, left sidebar) first.
#@markdown FLUX.1-schnell needs neither.

model_id = "black-forest-labs/FLUX.1-dev" #@param ["black-forest-labs/FLUX.1-dev", "black-forest-labs/FLUX.1-Krea-dev", "black-forest-labs/FLUX.1-schnell"] {allow-input: true}

import os
import subprocess
import sys
import time

# The token lives in Colab Secrets so it is never pasted into the notebook.
token = ""
try:
    from google.colab import userdata

    token = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    token = ""

if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN found in Colab Secrets.")
elif "schnell" in model_id:
    print("No HF_TOKEN - fine for FLUX.1-schnell, which is ungated.")
else:
    raise SystemExit(
        f"{model_id} is gated and no HF_TOKEN secret was found.\n"
        f"  1. Accept the licence at https://huggingface.co/{model_id}\n"
        "  2. Make a READ token at https://huggingface.co/settings/tokens\n"
        "  3. Colab left sidebar -> key icon -> Add new secret -> name HF_TOKEN,\n"
        "     paste the token, enable 'Notebook access'\n"
        "  4. Re-run this cell."
    )

# Only the diffusers-format folders. The repo also holds a single-file
# flux1-dev.safetensors (another 24 GB) that this pipeline never touches.
DIFFUSERS_ONLY = [
    "model_index.json",
    "scheduler/*",
    "tokenizer/*",
    "tokenizer_2/*",
    "text_encoder/*",
    "text_encoder_2/*",
    "transformer/*",
    "vae/*",
]


def fetch(disable_xet):
    # Run in a clean subprocess: huggingface_hub reads the Xet switch at import time.
    env = dict(os.environ)
    if disable_xet:
        env["HF_HUB_DISABLE_XET"] = "1"
    else:
        env.pop("HF_HUB_DISABLE_XET", None)
    code = (
        "from huggingface_hub import snapshot_download\n"
        "snapshot_download(%r, allow_patterns=%r, max_workers=8)\n" % (model_id, DIFFUSERS_ONLY)
    )
    return subprocess.call([sys.executable, "-c", code], env=env) == 0


# Hugging Face's CDN occasionally rejects its own signed URLs. Finished files are
# cached, so a retry only fetches what is still missing.
ok = False
for attempt in range(1, 4):
    print(f"\nDownloading {model_id} - plain HTTP (attempt {attempt}/3) ...")
    if fetch(disable_xet=True):
        ok = True
        break
    print("Hit a CDN error - retrying, already-downloaded shards are kept ...")
    time.sleep(5)

if not ok:
    print("\nFalling back to the Xet transfer path (can be slower on Colab) ...")
    ok = fetch(disable_xet=False)

if not ok:
    raise SystemExit(
        "Download failed. Common causes: the licence was not accepted for this repo, the token "
        "is not a READ token, the disk filled up, or huggingface.co is having an incident "
        "(https://status.huggingface.co). Fix and re-run - nothing is downloaded twice."
    )

print(f"\n{model_id} is cached and ready. Run cell 4.")

In [ ]:
#@title 4. Run the FLUX image generator { display-mode: "form" }
#@markdown Wait for the **`https://....gradio.live`** link, open it, and start generating.
#@markdown Leave this cell running - stopping it closes the app.

import os

try:
    from google.colab import userdata

    secret = (userdata.get("HF_TOKEN") or "").strip()
    if secret:
        os.environ["HF_TOKEN"] = secret
except Exception:
    pass

%cd /content/omnivoice-colab
!python flux_app.py

In [ ]:
#@title Optional: generate without the UI { display-mode: "form" }
#@markdown Prefer plain cells to a web UI? Stop cell 4 first (the app holds the GPU), then run this.

prompt = "a golden retriever puppy asleep on a stack of old books, warm window light, 85mm photograph, shallow depth of field" #@param {type:"string"}
model_id = "black-forest-labs/FLUX.1-dev" #@param ["black-forest-labs/FLUX.1-dev", "black-forest-labs/FLUX.1-Krea-dev", "black-forest-labs/FLUX.1-schnell"] {allow-input: true}
width = 1024 #@param {type:"slider", min:512, max:1536, step:64}
height = 1024 #@param {type:"slider", min:512, max:1536, step:64}
steps = 28 #@param {type:"slider", min:1, max:50, step:1}
guidance = 3.5 #@param {type:"slider", min:0, max:10, step:0.1}
seed = -1 #@param {type:"integer"}
num_images = 1 #@param {type:"slider", min:1, max:4, step:1}

import os
import random
import time

import torch
from diffusers import FluxPipeline
from IPython.display import display

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    pass

os.makedirs("/content/Flux_Output", exist_ok=True)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Reuse the pipeline across runs of this cell instead of reloading 34 GB every time.
if globals().get("_flux_pipe_id") != model_id:
    pipe = FluxPipeline.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, token=os.environ.get("HF_TOKEN") or None
    )
    if vram >= 34:
        pipe.to("cuda")
    else:
        pipe.enable_model_cpu_offload()
    globals()["_flux_pipe_id"] = model_id

max_seq = 256 if "schnell" in model_id else 512

for i in range(int(num_images)):
    this_seed = random.randint(0, 2**31 - 1) if int(seed) < 0 else int(seed) + i
    started = time.time()
    image = pipe(
        prompt=prompt,
        width=int(width) // 16 * 16,
        height=int(height) // 16 * 16,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance),
        max_sequence_length=max_seq,
        generator=torch.Generator("cpu").manual_seed(this_seed),
    ).images[0]
    path = f"/content/Flux_Output/flux_{this_seed}.png"
    image.save(path)
    print(f"seed {this_seed} - {time.time() - started:.1f}s - saved to {path}")
    display(image)

In [ ]:
#@title Optional: copy the generated images to Google Drive { display-mode: "form" }
#@markdown Colab wipes `/content` when the runtime disconnects. This copies everything to
#@markdown `MyDrive/Flux_Output` so it survives.

import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

source = Path("/content/Flux_Output")
target = Path("/content/drive/MyDrive/Flux_Output")
target.mkdir(parents=True, exist_ok=True)

copied = 0
for item in sorted(source.glob("*")):
    if item.is_file():
        shutil.copy2(item, target / item.name)
        copied += 1

print(f"Copied {copied} file(s) to {target}")

---

## Credits & licence

* Model: **[black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)** by Black Forest Labs
* Inference: [Hugging Face diffusers](https://github.com/huggingface/diffusers)
* Colab wrapper & Gradio app by [HiranNalaka](https://github.com/hirannalaka19)

**FLUX.1-dev** and **FLUX.1-Krea-dev** are released under the
[FLUX.1 \[dev\] Non-Commercial License](https://huggingface.co/black-forest-labs/FLUX.1-dev/blob/main/LICENSE.md) -
personal, research and evaluation use only, **no commercial use**.
**FLUX.1-schnell** is Apache-2.0 and may be used commercially.

## Usage disclaimer

You are responsible for what you generate. Do not use this to create sexual content involving
minors, non-consensual intimate imagery, deepfakes of real people intended to deceive,
harassment, disinformation, or anything else that is illegal where you live. Respect the
Black Forest Labs [Acceptable Use Policy](https://huggingface.co/black-forest-labs/FLUX.1-dev).